# 05 — Class-imbalance handling (fold-safe)

**Purpose.** Notebook 04 showed where the best models fail: **Ankylosing Spondylitis (AS) is under-predicted** (precision 0.735, recall 0.570) while **Reactive Arthritis (ReA) is over-predicted** (precision 0.558, recall 0.729). That is the signature of how class imbalance is handled. Here we compare five strategies under exactly the same 5 folds as notebook 04 and ask a single question: *does any strategy beat the current "balanced class weights" default by more than noise?*

**Reads:** `../data/raw/dataset.xlsx`, `../results/cv5_summary.csv` · **Writes:** `../results/imb_*.csv`, `imb_oof.npz`

**Placement in the pipeline.** Imbalance handling is evaluated before feature selection (notebook 06) because feature importance depends on how imbalance is handled (balanced weights, SMOTE or none change which features look useful), and the AS/ReA errors are the largest open gap after notebook 04.

**The five strategies**

| Code | Strategy | What it does | Why it is here |
|---|---|---|---|
| `none` | No handling | Plain training | Reference point: does imbalance handling help at all? |
| `balanced` | Balanced weights | Each patient weighted ∝ 1 / (their class frequency) → an ReA error costs ~5.5× an RA error | The current default in notebooks 03–04 (**the baseline to beat**) |
| `sqrt` | Square-root weights | Square root of the balanced weights (milder, ~2.3×) | Balanced weighting visibly *over*-corrects ReA (precision < recall); a half-strength correction may sit in between |
| `smotenc` | SMOTE-NC oversampling | Inside each training fold, synthetic minority patients are interpolated between real neighbours until all classes are the same size; Gender is treated as categorical | The standard alternative in the literature. Synthetic rows are created **only from training-fold patients and never evaluated** |
| `multipliers` | Post-hoc probability multipliers | Train with no weights, then multiply each class's predicted probability by a factor learned to maximise macro-F1 on **inner out-of-fold predictions** of the training fold | Directly attacks "wrong decision boundary" without changing the model. Fully nested: the validation patients are never used to learn the factors |

**Models:** Random Forest and XGBoost (the two strongest, most different in calibration). **Feature sets:** `raw` and `ext_noflags` — the two candidates left after notebook 04 (the missingness flags were dropped). Logistic Regression is clearly inferior and Gradient Boosting is ~35× slower with no accuracy advantage, so neither is included.

**Pre-specified decision rule (fixed before any results exist).**
* For each of the 4 combinations (2 models × 2 feature sets), a challenger replaces `balanced` only if the 95% paired-bootstrap interval of its macro-F1 gain **excludes 0** *and* it wins in **at least 4 of 5 folds**.
* A strategy is recommended **globally** only if it passes in **at least 3 of the 4 combinations**. If none does, `balanced` stays.
* Rationale: a rule tied to "the best model" can be decided by a negligible tie-break (as happened in notebook 04, where the top two models differed by 0.0014); requiring agreement across combinations removes that fragility.

## 0. Setup

**What this does:** imports libraries, makes `src/` importable, loads the shared evaluation helpers from `src/evaluation/metrics.py` (same functions as the inline ones in notebook 04), and prints versions.

In [1]:
import sys, os, time, warnings
sys.path.append('..')

import numpy as np
import pandas as pd
import sklearn, xgboost, imblearn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTENC

from src.preprocessing.cv_pipeline import preprocess_fold, load_cv_pool, EXT_NO_FLAGS
from src.evaluation.metrics import (macro_f1_fast, per_class_prf, brier_multiclass, ece_top_label,
                                    mcnemar_exact, holm, bootstrap_f1_matrix)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| imbalanced-learn", imblearn.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scikit-learn 1.9.0 | xgboost 3.4.1 | imbalanced-learn 0.14.2


## 1. Load the CV pool and verify it

**What this does:** loads only the 9,668 training patients (the test set is never loaded) and checks they are identical, in identical order, to notebook 02's training set. Nothing is imputed yet.

In [2]:
pool = load_cv_pool("../data/raw/dataset.xlsx")
ref_idx = pd.read_csv('../data/processed/y_train.csv', index_col=0).index
assert pool.shape == (9668, 17) and pool.index.equals(ref_idx)
y_global = LabelEncoder().fit_transform(pool['Disease'])
classes = sorted(pool['Disease'].unique()); K = len(classes)
AS, REA = classes.index('Ankylosing Spondylitis'), classes.index('Reactive Arthritis')
print("pool OK:", pool.shape, "| classes:", dict(enumerate(classes)))
print("focus classes -> AS =", AS, "| ReA =", REA)

pool OK: (9668, 17) | classes: {0: 'Ankylosing Spondylitis', 1: 'Normal', 2: 'Psoriatic Arthritis', 3: 'Reactive Arthritis', 4: 'Rheumatoid Arthritis', 5: "Sjögren's Syndrome", 6: 'Systemic Lupus Erythematosus'}
focus classes -> AS = 0 | ReA = 3


## 2. The same 5 folds as notebook 04

**What this does:** recreates `StratifiedKFold(5, shuffle=True, random_state=42)` — the identical folds used in notebook 04 — so results are paired with the baseline. Every patient is validated exactly once (checked).

In [3]:
N_SPLITS = 5
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(pool, pool['Disease']))
cover = np.zeros(len(pool), int)
for tr, va in folds:
    cover[va] += 1
assert (cover == 1).all()
print("fold sizes (train/val):", [(len(tr), len(va)) for tr, va in folds])

fold sizes (train/val): [(7734, 1934), (7734, 1934), (7734, 1934), (7735, 1933), (7735, 1933)]


## 3. Models, feature sets and the multiplier learner

**Models** use the same hyperparameters as notebook 04, except that Random Forest takes **no** `class_weight` argument here: every weighting is passed explicitly as `sample_weight`, so all strategies are applied the same way. (For Random Forest, `class_weight='balanced'` and `sample_weight=balanced` are numerically identical, and section 6 verifies that this notebook reproduces notebook 04's baseline exactly.)

**Multiplier learner.** Given predicted probabilities `P` (patients × 7) and true labels, it finds seven factors `w` so that predicting `argmax(P × w)` maximises macro-F1:
* start with all factors = 1;
* go class by class, trying 25 candidate factors between e^-1.5 ≈ 0.22 and e^1.5 ≈ 4.5, keep any change that improves macro-F1; repeat 3 passes.

This has only 7 parameters, so over-fitting is limited, and it is only ever fit on the training part of a fold.

**Honest caveat for `multipliers`:** the inner out-of-fold probabilities come from models trained on 2/3 of the training fold, but the imputer and scaler were fit on the whole training fold (including the inner validation rows). That makes the learned factors very slightly optimistic. It does **not** touch the outer validation patients, so the reported scores stay honest.

In [4]:
models = {
    'Random Forest': lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost':       lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                           objective='multi:softprob', eval_metric='mlogloss',
                                           random_state=42, n_jobs=-1),
}
feature_sets = ['raw', 'ext_noflags']
strategies = ['none', 'balanced', 'sqrt', 'smotenc', 'multipliers']

GRID = np.exp(np.linspace(-1.5, 1.5, 25))
def learn_multipliers(P, y, passes=3):
    w = np.ones(P.shape[1]); best = macro_f1_fast(y, (P * w).argmax(1))
    for _ in range(passes):
        for c in range(P.shape[1]):
            for g in GRID:
                w2 = w.copy(); w2[c] = g
                s = macro_f1_fast(y, (P * w2).argmax(1))
                if s > best + 1e-12:
                    best, w = s, w2
    return w

keys = [f"{m}|{f}|{s}" for m in models for f in feature_sets for s in strategies]
print(len(keys), "configurations x", N_SPLITS, "folds")

20 configurations x 5 folds


## 4. The cross-validation loop

**What this does, for each fold:**
1. Fit imputer/scaler/label encoder on the training part only (`preprocess_fold`), build the two feature sets.
2. For each model and feature set, train and score the five strategies on the **same** training part and the **same** validation patients:
   * `none`: fit without weights.
   * `balanced` / `sqrt`: fit with the corresponding per-patient weights.
   * `smotenc`: oversample the training part with SMOTE-NC (`k_neighbors=5`), fit without weights, predict real validation patients.
   * `multipliers`: reuse the `none` model's validation probabilities, multiply by factors learned from a 3-fold inner cross-validation inside the training part, renormalise.
3. Store each configuration's validation probabilities in a patient-aligned array (so after 5 folds every patient has exactly one out-of-fold prediction per configuration) and record fold metrics.

**Runtime:** roughly 12–20 minutes. One progress line per fold.

**Checks per fold:** no missing values after preprocessing; model classes are 0–6; encoded labels equal the global encoding.

In [5]:
oof_proba = {k: np.zeros((len(pool), K)) for k in keys}
y_all = np.full(len(pool), -1)
rows, mult_rows = [], []

def record(fold, mname, fname, strat, P, y_va_arr, va):
    key = f"{mname}|{fname}|{strat}"
    oof_proba[key][va] = P
    pred = P.argmax(1)
    pr, rc, f1 = per_class_prf(y_va_arr, pred, K)
    rows.append({'fold': fold, 'model': mname, 'features': fname, 'strategy': strat, 'key': key,
                 'macro_f1': f1_score(y_va_arr, pred, average='macro'),
                 'balanced_accuracy': rc.mean(), 'AS_f1': f1[AS], 'ReA_f1': f1[REA]})

for fold, (tr, va) in enumerate(folds, 1):
    t_fold = time.time()
    ft, fv = pool.iloc[tr], pool.iloc[va]
    Xr_tr, Xr_va, Xe_tr, Xe_va, y_tr, y_va = preprocess_fold(ft, fv)
    for Xd in (Xr_tr, Xr_va, Xe_tr, Xe_va):
        assert Xd.isna().sum().sum() == 0
    assert (y_va.to_numpy() == y_global[va]).all() and (y_tr.to_numpy() == y_global[tr]).all()
    ytr, yva = y_tr.to_numpy(), y_va.to_numpy()
    y_all[va] = yva
    sw_bal = compute_sample_weight('balanced', ytr); sw_sqrt = np.sqrt(sw_bal)
    sets = {'raw': (Xr_tr, Xr_va), 'ext_noflags': (Xe_tr[EXT_NO_FLAGS], Xe_va[EXT_NO_FLAGS])}
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for fname, (Xt, Xv) in sets.items():
            gi = list(Xt.columns).index('Gender')
            for mname, make in models.items():
                def fit(X, y, sw=None):
                    m = make(); m.fit(X, y, sample_weight=sw) if sw is not None else m.fit(X, y)
                    assert list(m.classes_) == list(range(K)); return m
                m_none = fit(Xt, ytr);              P_none = m_none.predict_proba(Xv)
                record(fold, mname, fname, 'none', P_none, yva, va)
                record(fold, mname, fname, 'balanced', fit(Xt, ytr, sw_bal).predict_proba(Xv), yva, va)
                record(fold, mname, fname, 'sqrt',     fit(Xt, ytr, sw_sqrt).predict_proba(Xv), yva, va)
                Xs, ys = SMOTENC(categorical_features=[gi], k_neighbors=5, random_state=42).fit_resample(Xt.to_numpy(), ytr)
                record(fold, mname, fname, 'smotenc', fit(Xs, ys).predict_proba(Xv.to_numpy()), yva, va)
                P_in = np.zeros((len(ytr), K))
                for itr, iva in StratifiedKFold(3, shuffle=True, random_state=1).split(Xt, ytr):
                    P_in[iva] = fit(Xt.iloc[itr], ytr[itr]).predict_proba(Xt.iloc[iva])
                w = learn_multipliers(P_in, ytr)
                P_mult = P_none * w; P_mult /= P_mult.sum(1, keepdims=True)
                record(fold, mname, fname, 'multipliers', P_mult, yva, va)
                mult_rows.append({'fold': fold, 'model': mname, 'features': fname, **{classes[c]: w[c] for c in range(K)}})
    print(f"fold {fold}/{N_SPLITS} done in {time.time() - t_fold:.0f}s")

fold_df = pd.DataFrame(rows)
oof_pred = {k: v.argmax(1) for k, v in oof_proba.items()}
print("rows in fold table:", len(fold_df))

fold 1/5 done in 48s
fold 2/5 done in 50s
fold 3/5 done in 50s
fold 4/5 done in 52s
fold 5/5 done in 52s
rows in fold table: 100


## 5. Integrity checks

**What this does:** every patient has a label and every probability row sums to 1 (nothing left unfilled), and the fast macro-F1 equals scikit-learn's on one configuration.

In [6]:
assert (y_all == y_global).all()
for k, P in oof_proba.items():
    assert np.allclose(P.sum(1), 1.0, atol=1e-6), k
k0 = 'Random Forest|ext_noflags|balanced'
a, b = f1_score(y_all, oof_pred[k0], average='macro'), macro_f1_fast(y_all, oof_pred[k0])
print(f"{k0}: sklearn {a:.6f} | fast {b:.6f}"); assert abs(a - b) < 1e-9
print("Integrity checks passed.")

Random Forest|ext_noflags|balanced: sklearn 0.815687 | fast 0.815687
Integrity checks passed.


## 6. Does the `balanced` baseline reproduce notebook 04?

**What this does:** compares this notebook's `balanced` results with `cv5_summary.csv` from notebook 04 for the same model and feature set. They should match to the 4th decimal: same folds, same preprocessing, same weights — only the code path differs (explicit `sample_weight` here, `class_weight` in 04 for Random Forest).

**If any difference exceeds 0.002, stop and investigate** — it would mean the two notebooks are not measuring the same thing and the paired comparisons below cannot be trusted.

In [7]:
ref = pd.read_csv('../results/cv5_summary.csv', index_col=0)['macro_f1_mean']
g = fold_df.groupby('key').macro_f1.mean()
cmp_rows = []
for m in models:
    for f in feature_sets:
        cmp_rows.append({'config': f"{m}|{f}", 'notebook_04': ref[f"{m}|{f}"], 'this_notebook': g[f"{m}|{f}|balanced"]})
cmp_df = pd.DataFrame(cmp_rows); cmp_df['diff'] = cmp_df['this_notebook'] - cmp_df['notebook_04']
print(cmp_df.round(5).to_string(index=False))
print("\nmax |diff| =", cmp_df['diff'].abs().max().round(5))
if cmp_df['diff'].abs().max() > 0.002:
    print("WARNING: baseline does not reproduce notebook 04 - investigate.")

                   config  notebook_04  this_notebook  diff
        Random Forest|raw      0.81449        0.81449   0.0
Random Forest|ext_noflags      0.81563        0.81563   0.0
              XGBoost|raw      0.81424        0.81424   0.0
      XGBoost|ext_noflags      0.81091        0.81091   0.0

max |diff| = 0.0


## 7. Results: all 20 configurations

**What this does:** one row per configuration, sorted by mean fold macro-F1.

* `macro_f1_mean ± std` — average and spread over the 5 folds (main number).
* `oof_macro_f1`, `ci_low`, `ci_high` — pooled out-of-fold score with a 95% bootstrap interval (2,000 resamples of patients).
* `AS_*` and `ReA_*` — F1, precision (`_p`) and recall (`_r`) for the two problem classes, from the pooled predictions.
* `brier`, `ece` — calibration. **Read with care:** weighting, SMOTE and multipliers all change the probabilities on purpose, so they are not expected to be calibrated for the real class mix; this is reported, not optimised.

**What to look for:** does any strategy move AS recall up and ReA precision up *without* hurting the rest? A strategy can trade one for the other and leave macro-F1 flat — that would show up here as AS and ReA moving in opposite directions.

In [8]:
keys_b, F = bootstrap_f1_matrix(y_all, oof_pred, B=2000, seed=42)
kpos = {k: j for j, k in enumerate(keys_b)}
gk = fold_df.groupby('key')
rows_s = []
for k in keys:
    pr, rc, f1 = per_class_prf(y_all, oof_pred[k], K)
    m, f, s = k.split('|')
    rows_s.append({'model': m, 'features': f, 'strategy': s,
                   'macro_f1_mean': gk.macro_f1.mean()[k], 'macro_f1_std': gk.macro_f1.std()[k],
                   'oof_macro_f1': macro_f1_fast(y_all, oof_pred[k]),
                   'ci_low': np.percentile(F[:, kpos[k]], 2.5), 'ci_high': np.percentile(F[:, kpos[k]], 97.5),
                   'AS_f1': f1[AS], 'AS_p': pr[AS], 'AS_r': rc[AS], 'ReA_f1': f1[REA], 'ReA_p': pr[REA], 'ReA_r': rc[REA],
                   'balanced_acc': gk.balanced_accuracy.mean()[k],
                   'brier': brier_multiclass(y_all, oof_proba[k]), 'ece': ece_top_label(y_all, oof_proba[k])})
summary = pd.DataFrame(rows_s, index=keys).sort_values('macro_f1_mean', ascending=False)
summary.round(4)

,model,features,strategy,macro_f1_mean,macro_f1_std,oof_macro_f1,ci_low,ci_high,AS_f1,AS_p,AS_r,ReA_f1,ReA_p,ReA_r,balanced_acc,brier,ece
Random Forest|ext_noflags|none,Random Forest,ext_noflags,none,0.8245,0.0098,0.8245,0.8154,0.8329,0.6667,0.7465,0.6022,0.6622,0.7455,0.5956,0.8156,0.2150,0.0246
Random Forest|raw|none,Random Forest,raw,none,0.8230,0.0070,0.8230,0.8139,0.8316,0.6623,0.7429,0.5975,0.6563,0.7845,0.5642,0.8114,0.2196,0.0370
Random Forest|ext_noflags|multipliers,Random Forest,ext_noflags,multipliers,0.8229,0.0075,0.8232,0.8142,0.8314,0.6669,0.7192,0.6216,0.6624,0.7019,0.6271,0.8176,0.2183,0.0234
Random Forest|ext_noflags|sqrt,Random Forest,ext_noflags,sqrt,0.8226,0.0097,0.8227,0.8140,0.8309,0.6549,0.7378,0.5887,0.6675,0.6691,0.6659,0.8216,0.2162,0.0271
Random Forest|raw|multipliers,Random Forest,raw,multipliers,0.8225,0.0079,0.8226,0.8136,0.8314,0.6580,0.7322,0.5975,0.6605,0.7233,0.6077,0.8159,0.2211,0.0398
Random Forest|raw|sqrt,Random Forest,raw,sqrt,0.8220,0.0060,0.8220,0.8132,0.8301,0.6436,0.7494,0.5640,0.6549,0.6852,0.6271,0.8187,0.2204,0.0432
XGBoost|ext_noflags|none,XGBoost,ext_noflags,none,0.8175,0.0083,0.8175,0.8091,0.8257,0.6630,0.6997,0.6298,0.6503,0.6961,0.6102,0.8120,0.2307,0.0583
XGBoost|raw|none,XGBoost,raw,none,0.8174,0.0057,0.8175,0.8090,0.8258,0.6673,0.7029,0.6351,0.6439,0.7040,0.5932,0.8108,0.2308,0.0575
XGBoost|raw|multipliers,XGBoost,raw,multipliers,0.8166,0.0081,0.8168,0.8081,0.8253,0.6578,0.6934,0.6257,0.6489,0.7198,0.5908,0.8088,0.2372,0.0689
XGBoost|raw|sqrt,XGBoost,raw,sqrt,0.8165,0.0028,0.8166,0.8081,0.8251,0.6609,0.6959,0.6293,0.6481,0.6473,0.6489,0.8158,0.2283,0.0563


## 8. Per-class F1 for each strategy, and the learned multipliers

**8a — Per-class F1 (pooled out-of-fold), Random Forest on `ext_noflags`** (the same table for XGBoost is saved to disk). Shows who gains and who loses when the strategy changes — a strategy that lifts AS by taking from Normal/Sjögren's is not an improvement.

**8b — Learned multipliers (mean over the 5 folds).** A factor > 1 means the strategy boosts that class's probability before predicting; < 1 means it suppresses it. If the multipliers say "boost AS, suppress ReA", that confirms the diagnosis from notebook 04 (AS under-predicted, ReA over-predicted) and shows the correction needed on top of an *unweighted* model.

In [9]:
pcf = {}
for m in models:
    for f in feature_sets:
        pcf[(m, f)] = pd.DataFrame({s: per_class_prf(y_all, oof_pred[f"{m}|{f}|{s}"], K)[2] for s in strategies}, index=classes).T
print("Per-class F1 - Random Forest | ext_noflags")
print(pcf[('Random Forest', 'ext_noflags')].round(3).to_string())
mult_df = pd.DataFrame(mult_rows)
print("\nLearned multipliers (mean over folds):")
print(mult_df.groupby(['model', 'features'])[classes].mean().round(2).to_string())

Per-class F1 - Random Forest | ext_noflags
             Ankylosing Spondylitis  Normal  Psoriatic Arthritis  Reactive Arthritis  Rheumatoid Arthritis  Sjögren's Syndrome  Systemic Lupus Erythematosus
none                          0.667   0.846                0.864               0.662                 0.861               0.886                         0.986
balanced                      0.642   0.848                0.872               0.632                 0.846               0.884                         0.985
sqrt                          0.655   0.848                0.865               0.667                 0.854               0.885                         0.985
smotenc                       0.636   0.847                0.863               0.620                 0.848               0.881                         0.984
multipliers                   0.667   0.844                0.862               0.662                 0.851               0.890                         0.986

Learned multip

## 9. Paired comparison against `balanced` and the pre-specified decision

**What this does:** for each of the 4 (model × feature set) combinations and each of the 4 challengers, compares the challenger to `balanced` on the same patients:

| Column | Meaning |
|---|---|
| `oof_diff` | pooled macro-F1 of the challenger minus `balanced` |
| `ci_low`, `ci_high` | 95% paired-bootstrap interval of that difference |
| `folds_better` | number of folds (of 5) where the challenger beats `balanced` |
| `AS_f1_diff`, `ReA_f1_diff` | change in F1 of the two problem classes |
| `mcnemar_b/c`, `p_holm` | McNemar exact test on accuracy (b = only challenger right, c = only baseline right), Holm-corrected over all 16 tests |
| `passes` | **the pre-specified rule**: interval excludes 0 on the positive side **and** ≥ 4 folds better |

Where the bootstrap interval and McNemar disagree, treat the result as borderline. The same caveat as notebook 04 applies: out-of-fold predictions come from overlapping training sets, so intervals and p-values are somewhat optimistic.

In [10]:
piv = fold_df.pivot(index='fold', columns='key', values='macro_f1')
out = []
for m in models:
    for f in feature_sets:
        base = f"{m}|{f}|balanced"
        for s in ['none', 'sqrt', 'smotenc', 'multipliers']:
            ch = f"{m}|{f}|{s}"
            d = F[:, kpos[ch]] - F[:, kpos[base]]
            bb, cc, p = mcnemar_exact(y_all, oof_pred[ch], oof_pred[base])
            fd = piv[ch] - piv[base]
            out.append({'model': m, 'features': f, 'challenger': s,
                        'oof_diff': macro_f1_fast(y_all, oof_pred[ch]) - macro_f1_fast(y_all, oof_pred[base]),
                        'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5),
                        'folds_better': int((fd > 0).sum()),
                        'AS_f1_diff': summary.loc[ch, 'AS_f1'] - summary.loc[base, 'AS_f1'],
                        'ReA_f1_diff': summary.loc[ch, 'ReA_f1'] - summary.loc[base, 'ReA_f1'],
                        'mcnemar_b': bb, 'mcnemar_c': cc, 'p_raw': p})
pairs = pd.DataFrame(out)
pairs['p_holm'] = holm(pairs['p_raw'])
pairs['passes'] = (pairs['ci_low'] > 0) & (pairs['folds_better'] >= 4)
pairs.drop(columns=['p_raw']).round(4)

,model,features,challenger,oof_diff,ci_low,ci_high,folds_better,AS_f1_diff,ReA_f1_diff,mcnemar_b,mcnemar_c,p_holm,passes
0,Random Forest,raw,none,0.0085,0.0018,0.0148,5,0.0314,0.0258,289,204,0.0021,True
1,Random Forest,raw,sqrt,0.0075,0.0023,0.0127,5,0.0127,0.0243,222,148,0.0021,True
2,Random Forest,raw,smotenc,-0.0047,-0.0099,0.0006,0,0.0007,-0.0278,194,220,0.6574,False
3,Random Forest,raw,multipliers,0.0082,0.0021,0.0138,5,0.0271,0.0300,259,195,0.0337,True
4,Random Forest,ext_noflags,none,0.0088,0.0027,0.0147,5,0.0243,0.0298,275,181,0.0002,True
5,Random Forest,ext_noflags,sqrt,0.0070,0.0025,0.0117,5,0.0126,0.0351,198,143,0.0339,True
6,Random Forest,ext_noflags,smotenc,-0.0042,-0.0090,0.0006,1,-0.0065,-0.0123,182,215,0.4945,False
7,Random Forest,ext_noflags,multipliers,0.0075,0.0018,0.0133,5,0.0245,0.0301,280,219,0.0645,True
8,XGBoost,raw,none,0.0033,-0.0030,0.0091,4,0.0194,-0.0061,303,241,0.0708,False
9,XGBoost,raw,sqrt,0.0023,-0.0029,0.0073,3,0.0130,-0.0019,259,220,0.4945,False


### 9b. Applying the rule

**What this does:** counts, for each challenger, in how many of the 4 combinations it passes, and applies the global rule (≥ 3 of 4). The strategy to carry into notebook 06 is printed at the end. Also printed: whether any challenger is *significantly worse* than `balanced` (interval entirely below 0), which would also be informative.

In [11]:
votes = pairs.groupby('challenger').agg(passes=('passes', 'sum'), mean_oof_diff=('oof_diff', 'mean'),
                                         mean_AS_diff=('AS_f1_diff', 'mean'), mean_ReA_diff=('ReA_f1_diff', 'mean'))
votes['significantly_worse_in'] = pairs.assign(w=pairs['ci_high'] < 0).groupby('challenger')['w'].sum()
print(votes.round(4).to_string())
winners = votes[votes['passes'] >= 3]
if len(winners):
    rec = winners['mean_oof_diff'].idxmax()
    print(f"\nRECOMMENDED strategy: {rec}  (passes in {int(votes.loc[rec, 'passes'])}/4 combinations)")
else:
    rec = 'balanced'
    print("\nRECOMMENDED strategy: balanced - no challenger passed the rule in >= 3 of 4 combinations.")

             passes  mean_oof_diff  mean_AS_diff  mean_ReA_diff  significantly_worse_in
challenger                                                                             
multipliers       2         0.0055        0.0161         0.0150                       0
none              3         0.0068        0.0225         0.0166                       0
smotenc           0        -0.0025        0.0012        -0.0139                       0
sqrt              2         0.0049        0.0114         0.0162                       0

RECOMMENDED strategy: none  (passes in 3/4 combinations)


## 10. Save the results

**What this does:** writes the tables and the out-of-fold probabilities (used later for stacking and error analysis) to `../results/`. Notebook 04's files are untouched.

In [12]:
fold_df.drop(columns=['key']).to_csv('../results/imb_fold_scores.csv', index=False)
summary.to_csv('../results/imb_summary.csv')
pairs.drop(columns=['p_raw']).to_csv('../results/imb_pairwise_vs_balanced.csv', index=False)
mult_df.to_csv('../results/imb_multipliers.csv', index=False)
np.savez_compressed('../results/imb_oof.npz', y=y_all, classes=np.array(classes),
                    **{k.replace('|', '__').replace(' ', '_'): v.astype(np.float32) for k, v in oof_proba.items()})
print(sorted(f for f in os.listdir('../results') if f.startswith('imb_')))

['imb_fold_scores.csv', 'imb_multipliers.csv', 'imb_oof.npz', 'imb_pairwise_vs_balanced.csv', 'imb_summary.csv']


## Summary — what this notebook establishes

* How five imbalance strategies compare against the balanced-weights default on Random Forest and XGBoost, under identical folds, with uncertainty and a decision rule fixed in advance.
* Whether the AS/ReA error pattern can be reduced, and at whose expense (section 8).
* The imbalance strategy to use from here on (printed in 9b). Hyperparameters are still untuned; feature selection (notebook 06) and nested-CV tuning (notebook 07) follow.